In [ ]:
# Setup
!pip install -q openai python-dotenv


## 1) Import


In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import json


## 2) Load environment variables - please read instructions carefully


In [ ]:
# if you are running in local, uncomment below line. also make sure you shall have a .env file
load_dotenv()

In [ ]:
# if you are running in google colab, uncomment below line. and replace "Your_API_Key" with your own openAI API key
#os.environ["OPENAI_API_KEY"] = "Your_API_Key"

In [ ]:
llm_name = "gpt-4o"
openai_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=openai_key)

## 3) LLM Call

In [ ]:
messages = []
message = {"role": "user", "content": "Plan a 2-day Tokyo trip for 2 adults. Mid comfort."}
messages.append(message)

In [ ]:
# Create an agent using OpenAI native tool calling
response = client.chat.completions.create(
    model=llm_name,
    temperature=0.4,
    messages=messages
)

In [ ]:
print("\n messages sent to LLM ")
print("\n", messages)

In [ ]:
print("\n response from LLM ")
print("\n", response)

In [ ]:
assistant_message = response.choices[0].message
print(assistant_message.content)

## 4) LLM Call - Structured output (prompt engineering)
**issues:**
- not reliable
- Notice how much of the prompt is about formatting instead of solving the actual task.

In [ ]:
messages = []
message = {"role": "user", "content": """Plan a 2-day Tokyo trip for 2 adults. Mid comfort. Output in JSON format, Return ONLY valid JSON.

The JSON must exactly follow this schema:
{
  "destination": string,
  "duration_days": integer,
  "travellers": integer,
  "comfort_level": string,
  "itinerary": [
    {
      "day": integer,
      "activities": [
        string
      ]
    }
  ]
}

Do not return markdown.
Do not wrap the JSON in ```json.
Do not include any additional text before or after the JSON."""
}
messages.append(message)

In [ ]:
# Create an agent using OpenAI native tool calling
response = client.chat.completions.create(
    model=llm_name,
    temperature=0.4,
    messages=messages
)

In [ ]:
assistant_message = response.choices[0].message
print(assistant_message.content)

## 5) LLM Call - Structured output (proper way)
- Returns valid structure.
- But still need to validate business rules.

In [ ]:
messages = []
message = {"role": "user", "content": "Plan a 2-day Tokyo trip for 2 adults. Mid comfort."}
messages.append(message)

In [ ]:
response = client.chat.completions.create(
    model=llm_name,
    temperature=0.4,
    messages=messages,
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "trip_plan",
            "schema": {
                "type": "object",
                "properties": {
                    "destination": {
                        "type": "string"
                    },
                    "duration_days": {
                        "type": "integer"
                    },
                    "travellers": {
                        "type": "integer"
                    },
                    "comfort_level": {
                        "type": "string"
                    },
                    "itinerary": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "day": {
                                    "type": "integer"
                                },
                                "activities": {
                                    "type": "array",
                                    "items": {
                                        "type": "string"
                                    }
                                }
                            },
                            "required": [
                                "day",
                                "activities"
                            ]
                        }
                    }
                },
                "required": [
                    "destination",
                    "duration_days",
                    "travellers",
                    "comfort_level",
                    "itinerary"
                ]
            }
        }
    }
)

In [ ]:
print("\nMessages sent to LLM")
print(messages)

In [ ]:
print("\nResponse")
print(response)

In [ ]:
assistant_message = response.choices[0].message
print(assistant_message.content)